In [1]:
import pandas as pd
import os 
import json

In [2]:
required_names = ["sample_id", "genotype_id", "qtl_group", "rna_qc_passed", "genotype_qc_passed", "study", "sex", "cell_type", "condition", "timepoint", "read_length", "stranded","paired","protocol"]

In [40]:
metadata_samples = f"/biodata/franco/datasets/geuvadis/E-GEUV-1.sdrf.txt"

## For GTEx. SEX: 1 = Male, 2 = Female
df_samples = pd.read_table(metadata_samples)
print(df_samples.columns)

keep_columns = ["Source Name", "Characteristics[sex]", 'Comment[SEQUENCE_LENGTH]',
       'Comment[LIBRARY_LAYOUT]', 'Factor Value[ancestry category]',
       'Factor Value[individual]' ]
crop_samples_df = df_samples[keep_columns].drop_duplicates().reset_index(drop=True)
crop_samples_df

crop_samples_df.to_csv("/biodata/franco/datasets/geuvadis/GEUVADIS_metadata_crop_for_qcnorm.txt", sep="\t", index=False)

Index(['Source Name', 'Comment[ENA_SAMPLE]', 'Characteristics[organism]',
       'Term Source REF', 'Term Accession Number',
       'Characteristics[individual]', 'Characteristics[sex]',
       'Characteristics[developmental stage]',
       'Characteristics[organism part]', 'Characteristics[cell type]',
       'Characteristics[cell line]', 'Characteristics[ancestry category]',
       'Comment[1000g Phase1 Genotypes]', 'Characteristics[laboratory]',
       'Protocol REF', 'Protocol REF.1', 'Extract Name',
       'Comment[LIBRARY_SELECTION]', 'Comment[LIBRARY_SOURCE]',
       'Comment[SEQUENCE_LENGTH]', 'Comment[LIBRARY_STRATEGY]',
       'Comment[LIBRARY_LAYOUT]', 'Comment[NOMINAL_LENGTH]',
       'Comment[NOMINAL_SDEV]', 'Protocol REF.2', 'Performer', 'Assay Name',
       'Technology Type', 'Comment[ENA_EXPERIMENT]',
       'Comment[READ_INDEX_1_BASE_COORD]', 'Protocol REF.3', 'Scan Name',
       'Comment[SUBMITTED_FILE_NAME]', 'Comment[ENA_RUN]',
       'Comment[FASTQ_URI]', 'Factor V

In [19]:
import gzip
rna_matrix_file = "/biodata/franco/datasets/geuvadis/reprocess/fsimonetti-nfdata-aws/FULL/featureCounts/merged_gene_counts.tsv.gz"
with gzip.open(rna_matrix_file, 'rt') as instream:
    line = instream.readline()
    sample_ids = line.rstrip().split("\t")[1:]
    print(len(sample_ids))

462


In [44]:
crop_samples_df

,Source Name,Characteristics[sex],Comment[SEQUENCE_LENGTH],Comment[LIBRARY_LAYOUT],Factor Value[ancestry category],Factor Value[individual]
0,HG00096,male,75,PAIRED,British,HG00096
1,HG00097,female,75,PAIRED,British,HG00097
2,HG00099,female,75,PAIRED,British,HG00099
3,HG00100,female,75,PAIRED,British,HG00100
4,HG00101,male,75,PAIRED,British,HG00101
...,...,...,...,...,...,...
457,NA20815,male,75,PAIRED,Tuscan,NA20815
458,NA20816,male,75,PAIRED,Tuscan,NA20816
459,NA20819,female,75,PAIRED,Tuscan,NA20819
460,NA20826,female,75,PAIRED,Tuscan,NA20826


In [54]:
#### Create qcnorm sample metadata file

df_sample_final = crop_samples_df[crop_samples_df["Source Name"].isin(sample_ids)]

df_sample_final = df_sample_final.drop(columns=['Comment[LIBRARY_LAYOUT]','Factor Value[ancestry category]', 'Factor Value[individual]'])
df_sample_final = df_sample_final.rename(columns={'Source Name':'sample_id','Characteristics[sex]':'sex', 'Comment[SEQUENCE_LENGTH]':'read_length', 'SEX':'sex'})

df_sample_final['genotype_id'] = df_sample_final['sample_id']
df_sample_final['cell_type'] = "LCL"
df_sample_final['qtl_group'] = "LCL"
df_sample_final['condition'] = "naive"
df_sample_final['timepoint'] = 0
#df_sample_final['read_length'] = "75bp"
df_sample_final['stranded'] = "TRUE"
df_sample_final['paired'] = "TRUE"
df_sample_final['protocol'] = "poly(A)"
df_sample_final['rna_qc_passed'] = "TRUE"
df_sample_final['genotype_qc_passed'] = "TRUE"
df_sample_final['study'] = "GEUVADIS"

df_sample_final["sex"] = df_sample_final["sex"].str.lower()

In [69]:
df_sample_final

,sample_id,sex,read_length,genotype_id,cell_type,qtl_group,condition,timepoint,stranded,paired,protocol,rna_qc_passed,genotype_qc_passed,study
0,HG00096,male,75,HG00096,LCL,LCL,naive,0,TRUE,TRUE,poly(A),TRUE,TRUE,GEUVADIS
1,HG00097,female,75,HG00097,LCL,LCL,naive,0,TRUE,TRUE,poly(A),TRUE,TRUE,GEUVADIS
2,HG00099,female,75,HG00099,LCL,LCL,naive,0,TRUE,TRUE,poly(A),TRUE,TRUE,GEUVADIS
3,HG00100,female,75,HG00100,LCL,LCL,naive,0,TRUE,TRUE,poly(A),TRUE,TRUE,GEUVADIS
4,HG00101,male,75,HG00101,LCL,LCL,naive,0,TRUE,TRUE,poly(A),TRUE,TRUE,GEUVADIS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
457,NA20815,male,75,NA20815,LCL,LCL,naive,0,TRUE,TRUE,poly(A),TRUE,TRUE,GEUVADIS
458,NA20816,male,75,NA20816,LCL,LCL,naive,0,TRUE,TRUE,poly(A),TRUE,TRUE,GEUVADIS
459,NA20819,female,75,NA20819,LCL,LCL,naive,0,TRUE,TRUE,poly(A),TRUE,TRUE,GEUVADIS
460,NA20826,female,75,NA20826,LCL,LCL,naive,0,TRUE,TRUE,poly(A),TRUE,TRUE,GEUVADIS


In [62]:
## check genotype samples
import gzip

with gzip.open("/biodata/franco/datasets/geuvadis/genotypes/latest_from1000G_high_coverage/GEUVADIS_1kGP_high_coverage.complete.MAF_0.01.vcf.gz", 'rt') as instream:
    for line in instream:
        if line.startswith("#CHROM"):
            gt_sampleids = line.rstrip().split()[9:]
            break
len(set.intersection(set(gt_sampleids),set(df_sample_final["sample_id"])))

449

In [64]:
## Samples in GEUVADIS but not in 1KGP 30x high coverage
missing_gt = set(df_sample_final["sample_id"])-set(gt_sampleids)
print(missing_gt)

{'HG00247', 'HG00135', 'HG00152', 'HG00312', 'NA20816', 'HG00377', 'HG00124', 'HG00134', 'HG00249', 'HG00156', 'HG00104', 'NA20537', 'HG00359'}


In [76]:
for sid in list(missing_gt):
    df_sample_final.loc[ df_sample_final["sample_id"] == sid,"genotype_qc_passed"] = "FALSE"
    print(df_sample_final[ df_sample_final["sample_id"] == sid ]["genotype_qc_passed"])

88    FALSE
Name: genotype_qc_passed, dtype: object
36    FALSE
Name: genotype_qc_passed, dtype: object
50    FALSE
Name: genotype_qc_passed, dtype: object
127    FALSE
Name: genotype_qc_passed, dtype: object
458    FALSE
Name: genotype_qc_passed, dtype: object
176    FALSE
Name: genotype_qc_passed, dtype: object
25    FALSE
Name: genotype_qc_passed, dtype: object
35    FALSE
Name: genotype_qc_passed, dtype: object
89    FALSE
Name: genotype_qc_passed, dtype: object
53    FALSE
Name: genotype_qc_passed, dtype: object
7    FALSE
Name: genotype_qc_passed, dtype: object
399    FALSE
Name: genotype_qc_passed, dtype: object
162    FALSE
Name: genotype_qc_passed, dtype: object


In [78]:
for i in required_names:
    if i not in df_sample_final.columns:
        print(i)
        raise
print("SUCCESS!")

df_sample_final.to_csv(f"/biodata/franco/datasets/geuvadis/GEUVADIS_sample_metadata_for_qcnorm.txt", sep="\t", index=False)

SUCCESS!
